In [0]:
from pyspark.sql.functions import lit, timestamp_diff, pow, radians, sin, cos, atan2, sqrt

In [0]:
df_trips = spark.read.table("bikes.02_silver.trips_cleansed")
df_stations = spark.read.table("bikes.02_silver.stations_cleansed")

In [0]:
def haversine_distance(lat1, lon1, lat2, lon2):
  EARTH_RADIUS = 3958.8

  lat1_rad = radians(lat1)
  lon1_rad = radians(lon1)
  lat2_rad = radians(lat2)
  lon2_rad = radians(lon2)

  dlat = lat2_rad - lat1_rad
  dlon = lon2_rad - lon1_rad

  a = pow(sin(dlat / 2), 2) + cos(lat1_rad) * cos(lat2_rad) * pow(
      sin(dlon / 2), 2
  )
  c = 2 * atan2(sqrt(a), sqrt(1 - a))

  return EARTH_RADIUS * c

In [0]:
df_join_start = df_trips.join(
    df_stations,
    (df_trips.start_station_id == df_stations.short_name) | 
    ((df_trips.start_lat == df_stations.latitude) & (df_trips.start_lng == df_stations.longitude)),
    "left"
).select(
    df_trips.city,
    df_trips.ride_id,
    df_trips.rideable_type,
    df_trips.started_at,
    df_trips.ended_at,
    df_stations.name.alias("start_station_name"),
    df_trips.end_station_id,
    df_stations.latitude.alias("start_latitude"),
    df_stations.longitude.alias("start_longitude"),
    df_trips.end_lat,
    df_trips.end_lng,
    df_trips.member_casual,
    df_trips.processed_timestamp
)

In [0]:
df_join_end = df_join_start.join(
    df_stations,
    (df_join_start.end_station_id == df_stations.short_name) |
    ((df_join_start.end_lat == df_stations.latitude) & (df_join_start.end_lng == df_stations.longitude)),
    "left"
).select(
    df_join_start.city,
    df_join_start.ride_id,
    df_join_start.rideable_type,
    df_join_start.started_at,
    df_join_start.ended_at,
    timestamp_diff(lit("MINUTE"),df_join_start.started_at,df_join_start.ended_at).alias("trip_duration_mins"),
    df_join_start.start_station_name,
    df_stations.name.alias("end_station_name"),
    df_join_start.start_latitude,
    df_join_start.start_longitude,
    df_stations.latitude.alias("end_latitude"),
    df_stations.longitude.alias("end_longitude"),
    haversine_distance(df_join_start.start_latitude,df_join_start.start_longitude,df_stations.latitude,df_stations.longitude).alias("stations_distance"),
    df_trips.member_casual,
    df_trips.processed_timestamp
)

In [0]:
df_join_end.filter(df_join_end.stations_distance > 0).write.mode("overwrite").saveAsTable("bikes.02_silver.trips_enriched")